01_Classification_Gemini.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/github/urcraft/data_analytics_lecture_notebooks/blob/main/01_Classification_Gemini.ipynb

<a target="_blank" href="https://colab.research.google.com/github/urcraft/data_analytics_lecture_notebooks/blob/main/01_Classification_Gemini.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Classification with Gemini: Support Ticket Routing

## What you will learn
- Use an LLM (Gemini) to classify text into business-relevant categories.
- Compare LLM classifications against ground-truth human labels.
- See where the model gets it right, where it fails, and why prompt wording matters.

**NLP Task**: Classification — assigning a label or category to a piece of text.

**Dataset**: `dair-ai/emotion` from HuggingFace — 6 emotion classes on short text.
We reframe it as a business scenario: routing customer messages by emotional tone.

Expected runtime: 5–10 minutes
Expected cost: Free-tier Gemini


In [ ]:
%pip install -q google-genai datasets pandas


In [ ]:
import os
import json
import time
import pandas as pd

# --- Gemini Setup ---
GEMINI_MODEL = 'gemini-3.1-flash-lite-preview'
print('Using Gemini model:', GEMINI_MODEL)

from google import genai

api_key = os.getenv('GOOGLE_API_KEY')
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('GOOGLE_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    raise ValueError('Set GOOGLE_API_KEY environment variable (or Colab secret).')

client = genai.Client(api_key=api_key)
print('Gemini client ready.')


## Load Dataset

We use the `dair-ai/emotion` dataset from HuggingFace.
It contains short English text labelled with one of 6 emotions:
sadness (0), joy (1), love (2), anger (3), fear (4), surprise (5).

We'll take a small sample of 20 examples to keep API calls fast and free-tier friendly.


In [ ]:
from datasets import load_dataset

ds = load_dataset('dair-ai/emotion', split='test')

LABEL_MAP = {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}

# Take a stratified sample: ~3-4 per class, 20 total
import random
random.seed(42)

indices_by_label = {}
for i, row in enumerate(ds):
    lab = row['label']
    indices_by_label.setdefault(lab, []).append(i)

sample_indices = []
for lab in sorted(indices_by_label.keys()):
    sample_indices.extend(random.sample(indices_by_label[lab], min(4, len(indices_by_label[lab]))))

sample_indices = sample_indices[:20]
sample_df = pd.DataFrame([ds[i] for i in sample_indices])
sample_df['ground_truth'] = sample_df['label'].map(LABEL_MAP)

print(f'Sample size: {len(sample_df)}')
print(f'Label distribution:\n{sample_df["ground_truth"].value_counts()}')
sample_df[['text', 'ground_truth']].head(8)


## Classify with Gemini

We ask Gemini to classify each message into one of the 6 emotion categories.
This simulates a business scenario: routing customer messages based on emotional tone
to decide priority and response style.


In [ ]:
VALID_LABELS = list(LABEL_MAP.values())

CLASSIFICATION_PROMPT = """You are a customer message router. Classify the emotional tone of the following message into exactly ONE of these categories:
{labels}

Message: "{text}"

Respond with ONLY the category label, nothing else."""

def classify_text(text: str) -> dict:
    prompt = CLASSIFICATION_PROMPT.format(labels=', '.join(VALID_LABELS), text=text)
    try:
        resp = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
        prediction = resp.text.strip().lower()
        # Clean up: take first word if model adds explanation
        if prediction not in VALID_LABELS:
            for label in VALID_LABELS:
                if label in prediction:
                    prediction = label
                    break
        return {'prediction': prediction, 'error': None}
    except Exception as e:
        return {'prediction': None, 'error': str(e)}

# Run classification on all samples
results = []
for idx, row in sample_df.iterrows():
    result = classify_text(row['text'])
    results.append(result)
    time.sleep(0.5)  # Respect rate limits

sample_df['prediction'] = [r['prediction'] for r in results]
sample_df['error'] = [r['error'] for r in results]
sample_df['correct'] = sample_df['prediction'] == sample_df['ground_truth']

print(f'\nAccuracy: {sample_df["correct"].mean():.1%}')
sample_df[['text', 'ground_truth', 'prediction', 'correct']].head(10)


## Analyze Errors

This is the critical step: understanding *where* and *why* the model fails.
In a business context, some misclassifications matter more than others.


In [ ]:
errors = sample_df[~sample_df['correct'] & sample_df['prediction'].notna()]
print(f'Total errors: {len(errors)} out of {len(sample_df)}')
print()

if len(errors) > 0:
    for _, row in errors.iterrows():
        print(f'Text: "{row["text"][:80]}..."')
        print(f'  Ground truth: {row["ground_truth"]}  |  Predicted: {row["prediction"]}')
        print()
else:
    print('No errors found! Try running again — LLM outputs can vary between runs.')


## Discussion Questions

1. **Which misclassifications would matter most in a business context?**
   - Classifying "anger" as "joy" is worse than confusing "love" with "joy"
   - What's the cost of each type of error in your domain?

2. **Would changing the category labels change the results?**
   - Try replacing "sadness" with "frustrated" or "disappointed"
   - This is the "lottery of prompting" from the lecture

3. **How would you evaluate this at scale?**
   - 20 examples is a demo. Production needs hundreds.
   - Which metrics matter? Overall accuracy? Per-class accuracy? Confusion matrix?

## Export for Student Annotation


In [ ]:
sample_df['student_notes'] = ''
export_path = 'classification_results.xlsx'
sample_df[['text', 'ground_truth', 'prediction', 'correct', 'student_notes']].to_excel(export_path, index=False)
print(f'Saved {export_path}')

try:
    from google.colab import files
    files.download(export_path)
except Exception:
    print('Download the file from the notebook working directory.')
